---
title: Alignment Metrics
subtitle: A view of alignment performance
date: "9999-12-31"
edit_url: null
---

In [ ]:
import altair as alt
import polars as pl
import os
from pathlib import Path

from harpy.report.html import StatsBox, print_html
from harpy.report.parsers import parse_samtools_stats
from harpy.report.plots import SafeRender
from harpy.report.tables import ITable
from harpy.report.utilities import StopExecution


In [ ]:
indir = "reports/data"
platform = "haplotagging"


In [ ]:
dat = parse_samtools_stats(indir, "raw")
if len(dat) == 0:
    print_html("No statsfiles were found, or were unable to be properly parsed. Skipping report.")
    raise StopExecution


## Unprocessed (raw) Alignments
The data presented below show the metrics for the alignments prior to duplicate identification, unmapped removal, quality filtering, etc. The summary boxes are averages across all the samples, and their standard deviations.

In [ ]:
_pp = dat["% Properly Paired"].replace(0, None).mean() or 0
_ppstd = dat["% Properly Paired"].replace(0, None).std() or 0

_mapped = dat["% Mapped"].replace(0, None).mean() or 0
_mappedstd = dat["% Mapped"].replace(0, None).std() or 0

#_smm = dat["% Self+Mate Mapped"].replace(0, None).mean() or 0
#_smmstd = dat["% Self+Mate Mapped"].replace(0, None).std() or 0

_isize = round(dat['Average Insert (bp)'].replace(0, None).mean() or 0)
_isizestd = round(dat['Average Insert (bp)'].replace(0, None).std() or 0)

_err = dat["Error Rate (%)"].replace(0, None).mean() or 0
_errstd = dat["Error Rate (%)"].replace(0, None).std() or 0

_pcr = dat["% Duplicates (PCR)"].mean() or 0
_pcrstd = dat["% Duplicates (PCR)"].std() or 0

_opt = dat["% Duplicates (Opt)"].mean() or 0
_optstd = dat["% Duplicates (Opt)"].std() or 0

(
    StatsBox()
    .add(len(dat), "Samples")
    .conditional(_pp, "Properly Paired", 80, plus_minus=_ppstd, add_percent=True, digits = 1)
    .conditional(_mapped, "Mapped", 50, plus_minus=_mappedstd, add_percent=True, digits = 1)
    .conditional(_err, "Error Rate", 10, plus_minus=_errstd, lower_bad = False, add_percent=True, digits = 3)
    .conditional(_pcr, "PCR Duplicate Rate", 40, plus_minus=_pcrstd, lower_bad = False, add_percent=True, digits = 2)
    .conditional(_opt, "Optical Duplicate Rate", 30, plus_minus=_optstd, lower_bad = False, add_percent=True, digits = 2)
    .add(_isize, "Insert Size", plus_minus=_isizestd, units = "bp")
).render()

:::{important} Duplicates
:class: dropdown
:open: true
Duplicates were marked after removing unmapped reads (unless `harpy align` ran with `-u`), which would mean the duplicate percents reported below would not include unmapped reads. The flow cell distance to determine optical duplicates was inferred from the sequencer ID at the start of sequence IDs.
:::

In [ ]:
os.makedirs(os.path.join(os.path.dirname(indir), 'tables'), exist_ok=True)
_ = dat.write_csv(os.path.join(os.path.dirname(indir), 'tables', 'align.raw.csv'))
ITable(dat, filename = "align.raw.stats", fixedcols = 1).render()

## Processed Alignments
The data presented below show the metrics for the alignments following post-processing, which includes duplicate identification, unmapped removal (unless configured otherwise), quality filtering, etc. The summary boxes are averages across all the samples, and their standard deviations.

In [ ]:
dat_filt = parse_samtools_stats(indir, "filtered")
if len(dat_filt) == 0:
    print_html("No statsfiles were found, or were unable to be properly parsed. Skipping report.")
    raise StopExecution
dat_filt = dat_filt.select(pl.col("*").exclude("% Duplicates (PCR)", "% Duplicates (Opt)"))

depths = []
for samplename in dat_filt['Sample']:
    df = pl.read_csv(
        os.path.join(indir, "coverage", f"{samplename}.regions.bed.gz"),
        separator='\t',
        has_header=False,
        new_columns=["Contig", "Position", "Position End", "Read Depth"],
        schema_overrides = {'Contig': pl.String}
    )
    depth = df.select(
        (
            (pl.col("Read Depth") * (pl.col("Position End") - pl.col("Position"))).sum()
            / (pl.col("Position End") - pl.col("Position")).sum()
        ).alias("depth")
    ).item()
    depths.append(round(depth, 2))
    #depths.append(round(df['Read Depth'].mean(), 2))

dat_filt = dat_filt.insert_column(2, pl.Series("Depth", depths))


In [ ]:
_pp = dat_filt["% Properly Paired"].replace(0, None).mean() or 0
_ppstd = dat_filt["% Properly Paired"].replace(0, None).std() or 0

_mapped = dat_filt["% Mapped"].replace(0, None).mean() or 0
_mappedstd = dat_filt["% Mapped"].replace(0, None).std() or 0

_depth = round(dat_filt["Depth"].replace(0, None).mean() or 0, 2)
_depthstd = round(dat_filt["Depth"].replace(0, None).std() or 0, 2)

_isize = round(dat_filt['Average Insert (bp)'].replace(0, None).mean() or 0)
_isizestd = round(dat_filt['Average Insert (bp)'].replace(0, None).std() or 0)

_err = dat_filt["Error Rate (%)"].replace(0, None).mean() or 0
_errstd = dat_filt["Error Rate (%)"].replace(0, None).std() or 0

(
    StatsBox()
    .add(len(dat_filt), "Samples")
    .conditional(_pp, "Properly Paired", 80, plus_minus=_ppstd, add_percent=True, digits = 1)
    .conditional(_mapped, "QC-Passing Alignments", 70, plus_minus=_mappedstd, add_percent=True, digits = 1)
    .add(_depth, "Coverage Depth", units = "X", plus_minus=_depthstd)
    .conditional(_err, "Error Rate", 10, plus_minus=_errstd, lower_bad = False, add_percent=True, digits = 3)
    .add(_isize, "Insert Size", plus_minus=_isizestd, units = "bp")
).render()

:::{hint} Usable Data
:class: dropdown
:open: true
All the values below are provided **ignoring reads marked as duplicates**, meaning these should be views as the properties of "usable data".
:::

In [ ]:
_ = dat_filt.write_csv(os.path.join(os.path.dirname(indir),'tables', 'align.processed.csv'))
ITable(dat_filt, filename = "align.processed.stats", fixedcols = 1).render()

## Duplication Rate
Duplication rate can vary significantly between libraries and preparation methods and it can often been a large pain-point in sequencing success. Below is a plot of duplication rate across samples, separated by PCR[^1] and Optical[^2] duplicates. Keep in mind these are kernel density estimates, so the Y-axis reflects density and not sample proportion. 

[^1]: PCR duplicates are DNA fragments identical to other ones due to PCR amplification. Unlike genomic duplicates, these reads are not biologically meaningful because they were created artificially through PCR.

[^2]: Optical duplicates are a machine reading error caused by the sequencer, where it incorrectly splits a single read into two or more identical ones.

In [ ]:
dupdata = dat.select(["Sample", "% Duplicates (PCR)", "% Duplicates (Opt)"]).rename(
    {"% Duplicates (PCR)" : "PCR", "% Duplicates (Opt)": "Optical"}
)

def density_chart(col):
    color = "#E69F00" if col == 'PCR' else "#56B4E9"
    return (
        alt.Chart(dupdata, width = 310, height = 250)
        .transform_density(density=col, extent=[0, 100], bandwidth=1)
        .mark_area(opacity=0.9, line=False, fill = color, interpolate="monotone")
        .encode(
            x=alt.X('value:Q', title=None).axis(format='d', labelExpr="datum.value + '%'"),
            y=alt.Y('density:Q', title='Density').axis(labels=False).stack(False),
            tooltip=[
                alt.Tooltip('value:Q', title = "Rate (%)"),
                alt.Tooltip("density:Q", title = "Density", format = '.2f')
            ]
        )
        .properties(title=f"{col} Duplicates")
    ).add_params(alt.selection_interval(bind="scales", encodings=["x"], name=f"brush_{col}"))

with SafeRender(dupdata):
    (
        alt.hconcat(density_chart('PCR'), density_chart('Optical'))
        .properties(
            usermeta={'embedOptions': {'downloadFileName': 'duplication'}}
        )
    ).display()